# Notebook 4 - City-Specific EDA and Baselines

## Airbnb Capstone | Madrid and Tokyo separately

**Input:** `airbnb_clean_final.csv`

This notebook follows the tutor recommendation to split the analysis by city before modelling. It creates separate Madrid and Tokyo datasets, exports city-specific EDA tables, and calculates simple baseline models that future machine-learning models should beat.

### Structure
1. Setup and helper functions
2. Split city datasets
3. Generate city-specific EDA tables
4. Run simple baseline models
5. Optional full model workflow
6. Output files


---
## 1. Setup, Imports, and Project Paths

In [3]:
# Optional Colab setup
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/Shareddrives/MBD_Capstone Project_KPMG/2. Code

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


def get_project_root() -> Path:
    """Find the CAPSTONE folder from any notebook subfolder."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        data_dir = candidate / "1. Data"
        if (data_dir / "master" / "airbnb_clean_final.csv").exists() or (data_dir / "raw" / "madrid_listings.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing the 1. Data folder. Check the notebook working directory.")


PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "1. Data"
MASTER_DIR = DATA_DIR / "master"
OUTPUT_DIR = DATA_DIR / "Outputs" / "city_specific"
CLEAN_DATA = MASTER_DIR / "airbnb_clean_final.csv"

TARGET = "price"
RANDOM_STATE = 42

# occupancy_rate is currently constant per city in the saved clean dataset, so
# keep the richer quarter/season/weekend occupancy features and exclude it.
KNOWN_UNSTABLE_FEATURES = {
    "occupancy_rate",
}

MODEL_CANDIDATE_FEATURES = [
    # Location
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    # Property
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "amenities_count",
    # Host
    "host_is_superhost",
    "host_response_rate",
    "host_acceptance_rate",
    "host_response_time",
    "host_listings_count",
    "instant_bookable",
    # Availability
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "minimum_nights",
    "maximum_nights",
    # Review and rating signals from listings.csv
    "number_of_reviews",
    "number_of_reviews_ltm",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "review_scores_value",
    "reviews_per_month",
    # Calendar features
    "occupancy_q1",
    "occupancy_q2",
    "occupancy_q3",
    "occupancy_q4",
    "occupancy_autumn",
    "occupancy_spring",
    "occupancy_summer",
    "occupancy_winter",
    "occupancy_weekday",
    "occupancy_weekend",
    # Review activity features; dynamically dropped if too sparse.
    "total_reviews",
    "days_since_last_review",
    "listing_age_days",
    "reviews_last_6m",
]

This cell imports the required libraries, detects the CAPSTONE project folder, and defines the shared constants used throughout the notebook.

---
## 2. Split City Datasets

These helper functions prepare the output folder, load the cleaned combined dataset, and split it into separate Madrid and Tokyo dataframes.

In [4]:
def ensure_output_dir() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
def load_clean_data() -> pd.DataFrame:
    if not CLEAN_DATA.exists():
        raise FileNotFoundError(f"Clean dataset not found: {CLEAN_DATA}")
    df = pd.read_csv(CLEAN_DATA)
    if "city" not in df.columns:
        raise ValueError("Expected a 'city' column in airbnb_clean_final.csv")
    return df

In [6]:
def split_city_data(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    cities = {}
    for city, city_df in df.groupby("city"):
        city_df = city_df.copy()
        cities[str(city)] = city_df
        out_file = MASTER_DIR / f"airbnb_{city.lower()}_clean.csv"
        city_df.to_csv(out_file, index=False)
    return cities

In [7]:
ensure_output_dir()

df = load_clean_data()
cities = split_city_data(df)

print("Combined dataset shape:", df.shape)
for city, city_df in cities.items():
    print(f"{city}: {city_df.shape[0]:,} rows, {city_df.shape[1]} columns")


Combined dataset shape: (41535, 46)
Madrid: 17,770 rows, 46 columns
Tokyo: 23,765 rows, 46 columns


This loads the previously cleaned combined dataset and writes separate Madrid and Tokyo clean datasets. These city-level files are useful for separate EDA and modelling.


---
## 3. Generate City-Specific EDA Tables

These helper functions calculate city-level summaries and save reusable EDA tables for neighbourhoods, room types, hosts, ratings, and prices.

In [8]:
def mode_or_missing(series: pd.Series) -> object:
    mode = series.mode(dropna=True)
    return mode.iloc[0] if len(mode) else np.nan

In [9]:
def top_n_counts(series: pd.Series, n: int = 5) -> str:
    counts = series.value_counts(dropna=False).head(n)
    return "; ".join(f"{idx}: {val}" for idx, val in counts.items())

In [10]:
def build_eda_summary(cities: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for city, df in cities.items():
        rows.append(
            {
                "city": city,
                "listings": len(df),
                "median_price_eur": df[TARGET].median(),
                "mean_price_eur": df[TARGET].mean(),
                "price_std_eur": df[TARGET].std(),
                "min_price_eur": df[TARGET].min(),
                "max_price_eur": df[TARGET].max(),
                "most_common_room_type": mode_or_missing(df["room_type"]),
                "top_neighbourhood": mode_or_missing(df["neighbourhood_cleansed"]),
                "top_5_neighbourhoods": top_n_counts(df["neighbourhood_cleansed"]),
                "superhost_rate_pct": df["host_is_superhost"].mean() * 100,
                "instant_bookable_rate_pct": df["instant_bookable"].mean() * 100,
                "mean_rating": df["review_scores_rating"].mean(),
                "mean_accommodates": df["accommodates"].mean(),
                "median_availability_365": df["availability_365"].median(),
                "missing_review_activity_pct": (
                    df["total_reviews"].isna().mean() * 100
                    if "total_reviews" in df.columns
                    else np.nan
                ),
            }
        )
    summary = pd.DataFrame(rows).round(3)
    summary.to_csv(OUTPUT_DIR / "city_specific_eda_summary.csv", index=False)
    return summary

In [11]:
def save_city_eda_tables(cities: dict[str, pd.DataFrame]) -> None:
    for city, df in cities.items():
        prefix = city.lower()
        price_by_room = (
            df.groupby("room_type")[TARGET]
            .agg(["count", "median", "mean", "std"])
            .sort_values("median", ascending=False)
            .round(3)
        )
        price_by_room.to_csv(OUTPUT_DIR / f"{prefix}_price_by_room_type.csv")

        neighbourhood = (
            df.groupby("neighbourhood_cleansed")
            .agg(
                listing_count=(TARGET, "size"),
                median_price_eur=(TARGET, "median"),
                mean_price_eur=(TARGET, "mean"),
                median_availability_365=("availability_365", "median"),
                mean_rating=("review_scores_rating", "mean"),
            )
            .query("listing_count >= 30")
            .sort_values("median_price_eur", ascending=False)
            .round(3)
        )
        neighbourhood.to_csv(OUTPUT_DIR / f"{prefix}_neighbourhood_summary.csv")

        numeric_corr = (
            df.select_dtypes(include=[np.number])
            .corr(numeric_only=True)[TARGET]
            .drop(labels=[TARGET], errors="ignore")
            .sort_values(key=lambda s: s.abs(), ascending=False)
            .round(4)
        )
        numeric_corr.to_csv(OUTPUT_DIR / f"{prefix}_price_correlations.csv")

In [12]:
eda_summary = build_eda_summary(cities)
save_city_eda_tables(cities)

eda_summary


,city,listings,median_price_eur,mean_price_eur,price_std_eur,min_price_eur,max_price_eur,most_common_room_type,top_neighbourhood,top_5_neighbourhoods,superhost_rate_pct,instant_bookable_rate_pct,mean_rating,mean_accommodates,median_availability_365,missing_review_activity_pct
0,Madrid,17770,105.000,114.599,62.300,8.0,305.000,Entire home/apt,Embajadores,Embajadores: 1878; Universidad: 1566; Palacio:...,25.250,49.195,4.649,3.136,244.0,100.000
1,Tokyo,23765,95.914,109.813,56.094,9.3,283.985,Entire home/apt,Shinjuku Ku,Shinjuku Ku: 4114; Sumida Ku: 3478; Toshima Ku...,40.964,78.426,4.744,4.259,164.0,13.057


This creates summary tables for each city, including price statistics, top neighbourhoods, superhost rates, instant-bookable rates, and rating averages. It also saves neighbourhood, room-type, and correlation tables to the output folder.


---
## 4. Run Simple Baseline Models

These baseline helpers create median-price reference models. They show what performance looks like before adding supervised machine learning.

In [13]:
def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    rmse = float(np.sqrt(np.mean((y_true_arr - y_pred_arr) ** 2)))
    mae = float(np.mean(np.abs(y_true_arr - y_pred_arr)))
    ss_res = float(np.sum((y_true_arr - y_pred_arr) ** 2))
    ss_tot = float(np.sum((y_true_arr - np.mean(y_true_arr)) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot else np.nan
    return {"rmse_eur": rmse, "mae_eur": mae, "r2": r2}

In [14]:
def train_baseline_models(cities: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Simple separate-city benchmarks that do not require scikit-learn."""
    rows = []
    rng = np.random.default_rng(RANDOM_STATE)

    for city, df in cities.items():
        model_df = df[
            ["neighbourhood_cleansed", "room_type", "accommodates", TARGET]
        ].dropna(subset=[TARGET]).copy()
        test_mask = rng.random(len(model_df)) < 0.20
        train = model_df.loc[~test_mask].copy()
        test = model_df.loc[test_mask].copy()

        city_median = train[TARGET].median()
        median_pred = np.repeat(city_median, len(test))
        rows.append(
            {
                "city": city,
                "model": "City median baseline",
                "rows": len(model_df),
                **regression_metrics(test[TARGET], median_pred),
            }
        )

        room_medians = train.groupby("room_type")[TARGET].median()
        room_pred = test["room_type"].map(room_medians).fillna(city_median)
        rows.append(
            {
                "city": city,
                "model": "Room type median baseline",
                "rows": len(model_df),
                **regression_metrics(test[TARGET], room_pred.to_numpy()),
            }
        )

        grouped = (
            train.groupby(["neighbourhood_cleansed", "room_type"])[TARGET]
            .median()
            .rename("segment_median")
            .reset_index()
        )
        segment_table = test.merge(
            grouped,
            on=["neighbourhood_cleansed", "room_type"],
            how="left",
        )
        segment_pred = (
            segment_table["segment_median"]
            .fillna(room_pred.reset_index(drop=True))
            .fillna(city_median)
        )
        rows.append(
            {
                "city": city,
                "model": "Neighbourhood + room median baseline",
                "rows": len(model_df),
                **regression_metrics(test[TARGET], segment_pred.to_numpy()),
            }
        )

        grouped_size = (
            train.groupby(["neighbourhood_cleansed", "room_type", "accommodates"])[
                TARGET
            ]
            .median()
            .rename("segment_median")
            .reset_index()
        )
        size_table = test.merge(
            grouped_size,
            on=["neighbourhood_cleansed", "room_type", "accommodates"],
            how="left",
        )
        size_pred = (
            size_table["segment_median"]
            .fillna(segment_pred.reset_index(drop=True))
            .fillna(city_median)
        )
        rows.append(
            {
                "city": city,
                "model": "Neighbourhood + room + size median baseline",
                "rows": len(model_df),
                **regression_metrics(test[TARGET], size_pred.to_numpy()),
            }
        )

    results = pd.DataFrame(rows).round(4)
    results.to_csv(OUTPUT_DIR / "city_specific_baseline_results.csv", index=False)
    return results

In [15]:
baseline_results = train_baseline_models(cities)
baseline_results


,city,model,rows,rmse_eur,mae_eur,r2
0,Madrid,City median baseline,17770,62.4519,48.7887,-0.0201
1,Madrid,Room type median baseline,17770,53.2498,38.6280,0.2584
2,Madrid,Neighbourhood + room median baseline,17770,50.0390,35.9383,0.3451
3,Madrid,Neighbourhood + room + size median baseline,17770,44.7261,31.1156,0.4768
4,Tokyo,City median baseline,23765,57.5502,43.0764,-0.0638
5,Tokyo,Room type median baseline,23765,56.5339,42.1229,-0.0266
6,Tokyo,Neighbourhood + room median baseline,23765,54.2382,39.7856,0.0551
7,Tokyo,Neighbourhood + room + size median baseline,23765,43.1832,30.5068,0.4010


These simple median-based baselines show what performance we can achieve before using machine learning. Future models should improve on these benchmarks.


---
## 5. Optional Full Model Workflow

These optional modelling helpers are kept near the optional training cell. They select candidate features, import modelling libraries, define preprocessing, and train city-specific ML models.

In [16]:
def candidate_features_for_city(
    df: pd.DataFrame, missing_threshold: float = 0.35
) -> tuple[list[str], list[str]]:
    existing = [
        col
        for col in MODEL_CANDIDATE_FEATURES
        if col in df.columns and col not in KNOWN_UNSTABLE_FEATURES
    ]
    missing_pct = df[existing].isna().mean()
    usable = missing_pct[missing_pct <= missing_threshold].index.tolist()

    categorical = [
        col
        for col in usable
        if df[col].dtype == "object" or str(df[col].dtype).startswith("category")
    ]
    numeric = [col for col in usable if col not in categorical]
    return numeric, categorical

In [17]:
def import_model_libraries():
    try:
        from sklearn.compose import ColumnTransformer
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import LinearRegression
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
        from sklearn.model_selection import train_test_split
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import OneHotEncoder, StandardScaler
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "Modelling requires scikit-learn. Run this script in Colab or install "
            "scikit-learn in the active Python environment."
        ) from exc

    try:
        from xgboost import XGBRegressor
    except ModuleNotFoundError:
        XGBRegressor = None

    return {
        "ColumnTransformer": ColumnTransformer,
        "LinearRegression": LinearRegression,
        "OneHotEncoder": OneHotEncoder,
        "Pipeline": Pipeline,
        "RandomForestRegressor": RandomForestRegressor,
        "SimpleImputer": SimpleImputer,
        "StandardScaler": StandardScaler,
        "XGBRegressor": XGBRegressor,
        "mean_absolute_error": mean_absolute_error,
        "mean_squared_error": mean_squared_error,
        "r2_score": r2_score,
        "train_test_split": train_test_split,
    }

In [18]:
def build_preprocessor(numeric: Iterable[str], categorical: Iterable[str], libs):
    numeric_pipeline = libs["Pipeline"](
        steps=[
            ("imputer", libs["SimpleImputer"](strategy="median")),
            ("scaler", libs["StandardScaler"]()),
        ]
    )
    categorical_pipeline = libs["Pipeline"](
        steps=[
            ("imputer", libs["SimpleImputer"](strategy="most_frequent")),
            ("onehot", libs["OneHotEncoder"](handle_unknown="ignore")),
        ]
    )
    return libs["ColumnTransformer"](
        transformers=[
            ("numeric", numeric_pipeline, list(numeric)),
            ("categorical", categorical_pipeline, list(categorical)),
        ]
    )

In [19]:
def model_specs(libs):
    models = {
        "Linear Regression": libs["LinearRegression"](),
        "Random Forest": libs["RandomForestRegressor"](
            n_estimators=250,
            min_samples_leaf=3,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }
    if libs["XGBRegressor"] is not None:
        models["XGBoost"] = libs["XGBRegressor"](
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    return models

In [20]:
def train_city_models(cities: dict[str, pd.DataFrame]) -> pd.DataFrame:
    libs = import_model_libraries()
    rows = []

    for city, df in cities.items():
        numeric, categorical = candidate_features_for_city(df)
        features = numeric + categorical
        model_df = df[features + [TARGET]].dropna(subset=[TARGET]).copy()

        x_train, x_test, y_train, y_test = libs["train_test_split"](
            model_df[features],
            np.log1p(model_df[TARGET]),
            test_size=0.20,
            random_state=RANDOM_STATE,
        )
        y_test_eur = np.expm1(y_test)

        for model_name, estimator in model_specs(libs).items():
            pipeline = libs["Pipeline"](
                steps=[
                    ("preprocessor", build_preprocessor(numeric, categorical, libs)),
                    ("model", estimator),
                ]
            )
            pipeline.fit(x_train, y_train)
            pred_eur = np.expm1(pipeline.predict(x_test))

            rmse = libs["mean_squared_error"](y_test_eur, pred_eur, squared=False)
            mae = libs["mean_absolute_error"](y_test_eur, pred_eur)
            r2 = libs["r2_score"](y_test_eur, pred_eur)

            rows.append(
                {
                    "city": city,
                    "model": model_name,
                    "rows": len(model_df),
                    "features": len(features),
                    "numeric_features": len(numeric),
                    "categorical_features": len(categorical),
                    "rmse_eur": rmse,
                    "mae_eur": mae,
                    "r2": r2,
                    "used_features": ", ".join(features),
                }
            )

    results = pd.DataFrame(rows).round(4)
    results.to_csv(OUTPUT_DIR / "city_specific_model_results.csv", index=False)
    return results

In [21]:
def main(run_models: bool = True) -> None:
    ensure_output_dir()
    df = load_clean_data()
    cities = split_city_data(df)

    summary = build_eda_summary(cities)
    save_city_eda_tables(cities)

    print("\nCity-specific EDA summary")
    print(summary.to_string(index=False))
    print(f"\nEDA outputs saved to: {OUTPUT_DIR}")

    baseline_results = train_baseline_models(cities)
    print("\nCity-specific baseline results")
    print(baseline_results.to_string(index=False))

    if run_models:
        try:
            results = train_city_models(cities)
        except RuntimeError as exc:
            print(f"\nSkipping model training: {exc}")
        else:
            print("\nCity-specific model results")
            print(results.drop(columns=["used_features"]).to_string(index=False))
            print(f"\nModel outputs saved to: {OUTPUT_DIR}")

In [22]:
# Run this only in an environment with scikit-learn installed.
# XGBoost is optional; if it is missing, the workflow will still run Linear Regression and Random Forest.

# model_results = train_city_models(cities)
# model_results.drop(columns=["used_features"])


This optional cell trains separate models for Madrid and Tokyo. It is commented out because some local Python environments do not include scikit-learn, but it can be run in Colab.


---
## 6. Output Files


In [23]:
print(OUTPUT_DIR)

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(path.name)


C:\Users\georg\Desktop\Docs\Student Files\CAPSTONE\1. Data\Outputs\city_specific
city_specific_baseline_results.csv
city_specific_eda_summary.csv
madrid_neighbourhood_summary.csv
madrid_price_by_room_type.csv
madrid_price_correlations.csv
tokyo_neighbourhood_summary.csv
tokyo_price_by_room_type.csv
tokyo_price_correlations.csv


This final cell lists the files created by the workflow so the team can quickly find the city-specific EDA and baseline outputs.
